In [13]:
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [14]:
from collections import Counter
import pandas as pd
import json



In [15]:
import pandas as pd
import json
from collections import Counter

# 1. Load data putusan yang sudah dilabeli
df = pd.read_csv('/content/drive/MyDrive/semester 6/Penalaran Komputer/UAS/data/eval/cases_labeled.csv')

# 2. Load hasil retrieval (top-5) dari file queries.json
with open('/content/drive/MyDrive/semester 6/Penalaran Komputer/UAS/data/eval/queries.json', 'r', encoding='utf-8') as f:
    queries = json.load(f)

# 3. Buat dictionary: {case_id: solusi}
case_solutions = dict(zip(df['case_id'], df['label_putusan']))

# 4. Fungsi prediksi berbasis majority vote DARI top_5_case_ids (tanpa memanggil retrieve)
def predict_outcome_from_top_cases(top_case_ids):
    solutions = [case_solutions.get(cid, "unknown") for cid in top_case_ids if case_solutions.get(cid)]
    if not solutions:
        return "Tidak ditemukan"
    return Counter(solutions).most_common(1)[0][0]

# 5. Proses semua query
results = []
for q in queries:
    query_id = q['query_id']
    query_text = q['query_text']
    top_cases = q['top_5_case_ids']
    predicted = predict_outcome_from_top_cases(top_cases)
    actual = case_solutions.get(q['ground_truth'], "unknown")
    results.append({
        "query_id": query_id,
        "predicted_solution": predicted,
        "actual_solution": actual,
        "top_5_case_ids": ', '.join(top_cases)
    })

# 6. Simpan ke DataFrame dan tampilkan
df_results = pd.DataFrame(results)

# 7. Tampilkan hasil awal
print("📊 Hasil Prediksi (Top 5):")
display(df_results.head())




📊 Hasil Prediksi (Top 5):


,query_id,predicted_solution,actual_solution,top_5_case_ids
0,Q115,ditahan,ditahan,"case_103, case_271, case_107, case_072, case_189"
1,Q302,ditahan,ditahan,"case_129, case_124, case_050, case_013, case_279"
2,Q289,ditahan,ditahan,"case_236, case_288, case_067, case_217, case_235"
3,Q199,ditahan,hukuman_mati,"case_194, case_005, case_142, case_163, case_042"
4,Q64,ditahan,ditahan,"case_052, case_217, case_067, case_179, case_042"


In [16]:
# 8. Hitung akurasi prediksi
df_results['is_correct'] = df_results['predicted_solution'] == df_results['actual_solution']
correct = df_results['is_correct'].sum()
total = len(df_results)
accuracy = correct / total * 100

# 9. Tampilkan hasil akurasi
print(f"\n📊 Jumlah Query Uji         : {total}")
print(f"🎯 Prediksi Benar           : {correct}")
print(f"✅ Akurasi Prediksi Amar    : {accuracy:.2f}%")



📊 Jumlah Query Uji         : 5
🎯 Prediksi Benar           : 4
✅ Akurasi Prediksi Amar    : 80.00%


In [17]:
import pandas as pd
import json
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity
from collections import Counter

# ===== 1. Load Data Case & Query =====
df = pd.read_csv('/content/drive/MyDrive/semester 6/Penalaran Komputer/UAS/data/eval/cases_labeled.csv').fillna("")

with open('/content/drive/MyDrive/semester 6/Penalaran Komputer/UAS/data/eval/queries.json', 'r', encoding='utf-8') as f:
    queries = json.load(f)

# Ambil maksimal 20 query uji
n_queries = min(20, len(queries))
queries_eval = queries[:n_queries]

# ===== 2. TF-IDF Vectorization (Fit on Train Set) =====
# Asumsikan seluruh data = data latih (jika belum di-split ulang)
train_texts = df['text_full']
train_case_ids = df['case_id'].tolist()

# Stopwords bahasa Indonesia
stopwords_indonesia = [
    'yang', 'dan', 'di', 'ke', 'dari', 'akan', 'karena', 'bahwa', 'untuk', 'dengan',
    'pada', 'adalah', 'itu', 'ini', 'dalam', 'tidak', 'oleh', 'sebagai', 'juga', 'atau',
    'sudah', 'sangat', 'masih', 'lagi', 'lebih', 'hanya', 'maka', 'bagi', 'antara'
]

tfidf_vectorizer = TfidfVectorizer(stop_words=stopwords_indonesia, max_features=1000)
train_vectors = tfidf_vectorizer.fit_transform(train_texts)

# Dictionary case_id -> label_putusan
case_solutions = dict(zip(df['case_id'], df['label_putusan']))

# ===== 3. Majority Vote Function =====
def predict_majority_vote_from_top_cases(top_case_ids):
    solutions = [case_solutions.get(cid, "unknown") for cid in top_case_ids if case_solutions.get(cid)]
    if not solutions:
        return "Tidak ditemukan"
    return Counter(solutions).most_common(1)[0][0]

# ===== 4. Weighted Similarity Function =====
def predict_weighted_vote(query_text: str):
    query_vec = tfidf_vectorizer.transform([query_text])
    similarities = cosine_similarity(query_vec, train_vectors).flatten()
    top_k_idx = similarities.argsort()[::-1][:5]

    solution_weights = {}
    for idx in top_k_idx:
        cid = train_case_ids[idx]
        solution = case_solutions.get(cid)
        if solution:
            score = similarities[idx]
            solution_weights[solution] = solution_weights.get(solution, 0) + score

    if not solution_weights:
        return "Tidak ditemukan"

    return max(solution_weights.items(), key=lambda x: x[1])[0]

# ===== 5. Evaluasi dan Perbandingan =====
results = []
for q in queries_eval:
    qid = q['query_id']
    qtext = q['query_text']
    actual = case_solutions.get(q['ground_truth'], "unknown")

    # Majority vote dari top-5 yang sudah disediakan
    pred_major = predict_majority_vote_from_top_cases(q['top_5_case_ids'])

    # Weighted similarity dari vektorisasi ulang
    pred_weight = predict_weighted_vote(qtext)

    results.append({
        "query_id": qid,
        "actual_solution": actual,
        "pred_majority": pred_major,
        "pred_weighted": pred_weight,
        "top_5_case_ids": ', '.join(q['top_5_case_ids'])
    })

df_results = pd.DataFrame(results)

# ===== 6. Hitung Akurasi =====
acc_major = (df_results['actual_solution'] == df_results['pred_majority']).mean() * 100
acc_weight = (df_results['actual_solution'] == df_results['pred_weighted']).mean() * 100

# ===== 7. Tampilkan Hasil =====
print(f"📊 Jumlah Query Uji              : {n_queries}")
print(f"🎯 Akurasi Majority Vote         : {acc_major:.2f}%")
print(f"⚖️  Akurasi Weighted Similarity  : {acc_weight:.2f}%")

df_results.head()


📊 Jumlah Query Uji              : 5
🎯 Akurasi Majority Vote         : 80.00%
⚖️  Akurasi Weighted Similarity  : 80.00%


,query_id,actual_solution,pred_majority,pred_weighted,top_5_case_ids
0,Q115,ditahan,ditahan,ditahan,"case_103, case_271, case_107, case_072, case_189"
1,Q302,ditahan,ditahan,ditahan,"case_129, case_124, case_050, case_013, case_279"
2,Q289,ditahan,ditahan,ditahan,"case_236, case_288, case_067, case_217, case_235"
3,Q199,hukuman_mati,ditahan,ditahan,"case_194, case_005, case_142, case_163, case_042"
4,Q64,ditahan,ditahan,ditahan,"case_052, case_217, case_067, case_179, case_042"


In [18]:
# 6. Simpan ke file
results_path = '/content/drive/MyDrive/semester 6/Penalaran Komputer/UAS/data/eval'
os.makedirs(results_path, exist_ok=True)
df_results.to_csv(os.path.join(results_path, 'predictions.csv'), index=False)